# Ridge regression reference

Train the four registered CPU ridge settings through shared Python modules. Scaling uses training only; validation selects alpha. Calibration and final test remain locked. No model code lives in this notebook.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import sys
import pandas as pd

here = Path.cwd().resolve()
repository = next(path for path in [here, *here.parents] if (path / 'docs/data/prepared-release.json').is_file())
if str(repository) not in sys.path:
    sys.path.insert(0, str(repository))
from ml.training.ridge import run_ridge

In [ ]:
run_id = datetime.now(timezone.utc).strftime('ridge-notebook-%Y%m%dT%H%M%S%fZ')
output = repository / 'ml/runs' / run_id
report = run_ridge(
    repository / 'ml/training/configs/do_pilot.json',
    repository / 'ml/training/configs/ridge_settings.json',
    output, repository=repository, matrix=True,
)

In [ ]:
pd.DataFrame([{
    'configuration': case['configuration'],
    'horizon_hours': case['horizon_hours'],
    'selected_alpha': case.get('selected_alpha'),
    'validation_mae_mgL': case['selected']['validation']['mae_mgL'] if case['selected'] else None,
    'validation_error_gates_passed': case.get('validation_error_gates', {}).get('passed', False),
    'model_qualified': case.get('model_qualified', False),
} for case in report['cases']])